# KPX 한국전력거래소 — 시간별 전국 전력수요량 EDA

In [ ]:
import sys
sys.path.insert(0, "..")  # 루트 config.py 접근

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from config import DATA_DIR, DATE_COL, HOUR_COLS, CSV_ENCODING

plt.rcParams["font.family"] = "AppleGothic"   # macOS 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

## 1. 데이터 로드 및 전처리

In [ ]:
csv_path = next(DATA_DIR.glob("*.csv"))
print(f"파일: {csv_path.name}")

wide = pd.read_csv(csv_path, encoding=CSV_ENCODING, parse_dates=[DATE_COL])
wide[HOUR_COLS] = wide[HOUR_COLS].apply(pd.to_numeric, errors="coerce")
print(wide.shape)
wide.head(3)

In [ ]:
# wide → long (날짜+시간 → 단일 datetime 인덱스)
long = wide.melt(id_vars=DATE_COL, value_vars=HOUR_COLS, var_name="시간", value_name="수요량(MW)")
long["hour"] = long["시간"].str.replace("시", "").astype(int) - 1  # 0~23
long["datetime"] = long[DATE_COL] + pd.to_timedelta(long["hour"], unit="h")
long = long.sort_values("datetime").set_index("datetime")[["수요량(MW)"]]

print(f"총 {len(long)}개 포인트 | {long.index.min()} ~ {long.index.max()}")
long.describe()

## 2. 전체 시계열

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(long.index, long["수요량(MW)"], linewidth=0.6, color="steelblue")
ax.set_title("시간별 전국 전력수요량 (전체)")
ax.set_ylabel("MW")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

## 3. 일간 주기 패턴 (시간대별 평균)

In [ ]:
hourly_mean = long.groupby(long.index.hour)["수요량(MW)"].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_mean.index, hourly_mean.values, marker="o", color="tomato")
ax.set_title("시간대별 평균 전력수요 (일간 주기)")
ax.set_xlabel("시각 (0~23시)")
ax.set_ylabel("MW")
ax.set_xticks(range(24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 주간 주기 패턴 (요일별 평균)

In [ ]:
day_labels = ["월", "화", "수", "목", "금", "토", "일"]
daily_mean = long.groupby(long.index.dayofweek)["수요량(MW)"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(day_labels, daily_mean.values, color=["steelblue"]*5 + ["salmon"]*2)
ax.set_title("요일별 평균 전력수요 (주간 주기)")
ax.set_ylabel("MW")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 히트맵 (요일 × 시간대)

In [ ]:
pivot = long.copy()
pivot["hour"] = pivot.index.hour
pivot["dayofweek"] = pivot.index.dayofweek
heatmap = pivot.groupby(["dayofweek", "hour"])["수요량(MW)"].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap.values, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(7))
ax.set_yticklabels(day_labels)
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h}시" for h in range(24)], rotation=45, fontsize=8)
ax.set_title("요일 × 시간대별 평균 전력수요 히트맵")
plt.colorbar(im, ax=ax, label="MW")
plt.tight_layout()
plt.show()

## 6. 월별 평균 (계절 패턴)

In [ ]:
monthly = long.resample("ME")["수요량(MW)"].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly.index.strftime("%Y-%m"), monthly.values, color="mediumseagreen")
ax.set_title("월별 평균 전력수요 (계절 패턴)")
ax.set_ylabel("MW")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()